> 在线讲义：[第 24 章 · 大厂怎么选配置：LLaMA 3 的缩放律选型、长上下文续训、Qwen3 的 QK-norm](https://wikieden.github.io/beyond-transformer/lessons/24-configs.html) · 本书《超越 Transformer》

# 第 24 章 · 大厂怎么选配置：LLaMA 3 的缩放律选型、长上下文续训、Qwen3 的 QK-norm

配套讲义：[在线版](https://wikieden.github.io/beyond-transformer/lessons/24-configs.html) · 仓库里 `beyond-transformer/notes/lessons/24-configs.md`（附录最后一节"代码对照"）

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的 Python 写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。只用 NumPy，不需要 GPU。练习题不在这里算，先在讲义里自己做。

In [ ]:
import numpy as np

np.set_printoptions(precision=4, suppress=True)  # 打印保留 4 位小数，不用科学计数法
print("numpy", np.__version__)

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一个函数、跑出来是什么。

| 讲义 | 公式 | 写法 | 跑出来 |
|---|---|---|---|
| 第一步 缩放律 | A·C^α、C/(6D) | `optimal_tokens`、`params_from` | 16.29T / 389B；10.46T / 605B |
| 第二步 token / 参数 | D/N | `RUNS` | 20、149、1875、39 |
| 第三步 配置表 | 三个比例 | `read_config(LLAMA3[...])` | 128、3.5、4 |
| 第四步 分段 | 等比 | `context_stages()` | 8192 … 131072 |

token / 参数比从 Chinchilla 的约 20 一路涨到上千；FFN 宽度比、GQA 头数这些配置怎么读；长上下文分六段续训；
Qwen3 用 QK-norm 换掉 QKV bias。

## 1. 先把要用的东西定义好

这几格只是定义，运行后没有输出也正常。

In [ ]:
C_FLAG = 3.8e25

**函数 `optimal_tokens`**：LLaMA 3 §3.2.1：N*(C) = A·C^α，N* 是最优训练 token 数（论文里这个 N 指 token，不是参数）。

In [ ]:
def optimal_tokens(c, a=0.29, alpha=0.53):
    """LLaMA 3 §3.2.1：N*(C) = A·C^α，N* 是最优训练 token 数（论文里这个 N 指 token，不是参数）。"""
    return a * c**alpha

**函数 `params_from`**：训练 FLOPs ≈ 6 × 参数 × token（第 8 章）⇒ 参数 ≈ C / (6·token)。

In [ ]:
def params_from(c, tokens):
    """训练 FLOPs ≈ 6 × 参数 × token（第 8 章）⇒ 参数 ≈ C / (6·token)。"""
    return c / (6 * tokens)

In [ ]:
RUNS = {  # (参数, 训练 token)：公开报告里的数
    "Chinchilla 70B": (70e9, 1.4e12),
    "LLaMA 1 7B": (6.7e9, 1.0e12),
    "LLaMA 3 8B": (8e9, 15e12),
    "LLaMA 3 405B": (405e9, 15.6e12),
}
LLAMA3 = {  # 论文表 3（L322–L333）
    "8B": dict(layers=32, d=4096, ffn=14336, heads=32, kv=8),
    "70B": dict(layers=80, d=8192, ffn=28672, heads=64, kv=8),
    "405B": dict(layers=126, d=16384, ffn=53248, heads=128, kv=8),
}
QWEN3 = {"Qwen3-0.6B": (28, 16, 8, "是", "32K"), "Qwen3-8B": (36, 32, 8, "否", "128K"), "Qwen3-32B": (64, 64, 8, "否", "128K")}

**函数 `read_config`**：小工具。

In [ ]:
def read_config(c):
    return {"FFN / d": c["ffn"] / c["d"], "每头维数": c["d"] // c["heads"], "Q 头 / KV 头": c["heads"] // c["kv"]}

**函数 `context_stages`**：长上下文续训：从 8K 分 6 段涨到 128K（每段乘同一个倍数，示意）。

In [ ]:
def context_stages(start=8192, end=131072, stages=6):
    """长上下文续训：从 8K 分 6 段涨到 128K（每段乘同一个倍数，示意）。"""
    r = (end / start) ** (1 / (stages - 1))
    return [round(start * r**i) for i in range(stages)]

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
print("第一步 LLaMA 3 的缩放律选型")
for a, al in ((0.29, 0.53), (0.299, 0.537)):
    t = optimal_tokens(C_FLAG, a, al)
    print(f"  (A, α) = ({a}, {al})：C = 3.8e25 → 最优 token {t / 1e12:.2f}T，参数 ≈ C/(6·token) = {params_from(C_FLAG, t) / 1e9:.0f}B")
print("  论文：402B 参数、16.55T token；实际训的是 405B、15.6T")
print("\n第二步 token / 参数比")
for k, (n, d) in RUNS.items():
    print(f"  {k}：{d / n:,.0f} 个 token / 参数，训练 FLOPs ≈ {6 * n * d:.2e}")

In [ ]:
print("\n第三步 读配置表")
for k, c in LLAMA3.items():
    print(f"  LLaMA 3 {k}：{read_config(c)}")
for k, (l, h, kv, tie, ctx) in QWEN3.items():
    print(f"  {k}：{l} 层，Q/KV 头 {h}/{kv}，共享嵌入 {tie}，上下文 {ctx}")

In [ ]:
print("\n第四步 长上下文分段续训（示意）")
print(f"  6 段：{context_stages()}")

## 3. 代码里才看得到的

- **论文里的 N 指 token**：LLaMA 3 写的 N*(C) 是“最优训练 token 数”，不是参数量；和 Chinchilla 的符号习惯不同，读的时候很容易搞混。
- **配置表只用来读比例**：用这些配置逐项数参数、对上官方的 8B / 70B / 405B，是里程碑 T-M1。

## 4. 自己改一改

- 把 C 换成 10²³、10²⁷，两组拟合参数的推荐相差多少倍？
- 把 `context_stages` 改成 4 段，每段乘多少？

在下面的空格子里写，或者直接改上面的格子。

In [ ]:
# 在这里写你的代码

## 口诀

> 小锅试做找最优，点连成线外推大锅；指数差一点，外推差一截。
> 旗舰按缩放律，小模型多训数据，一个参数喂上千个 token。
> 读配置：头维一二八，FFN 三倍半，KV 固定八组，RoPE 底数调大。
> 长上下文放最后，分段拉长；Qwen3 去掉 QKV bias 换 QK-norm。